In [1]:
import fastf1
import pandas as pd
import os
import numpy as np
import gc

In [2]:
os.makedirs('./cache', exist_ok=True)
fastf1.Cache.enable_cache('./cache')

Normalised Lap time detection and retrival

In [3]:
years = [2023, 2024,2025]
track = 'Bahrain'

for year in years:
    print(f"\nProcessing {track} {year}")

    try:
        session = fastf1.get_session(year, track, 'R')
        session.load()

        laps = session.laps[
            ['Driver', 'LapNumber', 'LapTime', 'Compound', 'TyreLife', 'Stint']
        ].copy()

        laps['LapTimeSeconds'] = laps['LapTime'].dt.total_seconds()
        laps.drop(columns=['LapTime'], inplace=True)


        laps = laps.dropna(subset=['LapTimeSeconds', 'TyreLife'])
        laps = laps[laps['TyreLife'] > 0]

        mean = laps['LapTimeSeconds'].mean()
        std = laps['LapTimeSeconds'].std()

        laps = laps[laps['LapTimeSeconds'] < (mean + 2 * std)]

   
        k = 0.04  # seconds per lap loosing due to fuel usage. This value lies between 0.03s-0.05s. So we take the mean of the same. 
        laps['FuelCorrectedLapTime'] = (
            laps['LapTimeSeconds'] - k * laps['LapNumber']
        )
        
        fastest = laps['FuelCorrectedLapTime'].min()
        laps['NormalizedLapTime'] = laps['FuelCorrectedLapTime'] - fastest


        laps['DeltaLapTime'] = laps.groupby(
            ['Driver', 'Stint']
        )['FuelCorrectedLapTime'].transform(
            lambda x: x - x.iloc[0]
        )


        laps['Driver'] = laps['Driver'].astype('category')
        laps['Compound'] = laps['Compound'].astype('category')

        laps['LapNumber'] = laps['LapNumber'].astype('int16')
        laps['TyreLife'] = laps['TyreLife'].astype('int16')
        laps['Stint'] = laps['Stint'].astype('int8')

        laps['LapTimeSeconds'] = laps['LapTimeSeconds'].astype('float32')
        laps['FuelCorrectedLapTime'] = laps['FuelCorrectedLapTime'].astype('float32')
        laps['NormalizedLapTime'] = laps['NormalizedLapTime'].astype('float32')
        laps['DeltaLapTime'] = laps['DeltaLapTime'].astype('float32')

        file_path = f"data/{track.lower()}_{year}.parquet"
        laps.to_parquet(file_path, index=False)

        print(f"Saved → {file_path}")
        print(laps.head())

        del laps
        gc.collect()

    except Exception as e:
        print(f"Error in {year}: {e}")
        print(laps)


Processing Bahrain 2023


core           INFO 	Loading data for Bahrain Grand Prix - Race [v3.8.1]
req            INFO 	No cached data found for session_info. Loading data...
_api           INFO 	Fetching session info data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for driver_info. Loading data...
_api           INFO 	Fetching driver list...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for session_status_data. Loading data...
_api           INFO 	Fetching session status data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for lap_count. Loading data...
_api           INFO 	Fetching lap count data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for track_status_data. Loading data...
_api           INFO 	Fetching track status data...
req            INFO 	Data has been written to cache!
req            INFO 	No ca

Error in 2023: Cannot save file into a non-existent directory: 'data'
     Driver  LapNumber Compound  TyreLife  Stint  LapTimeSeconds  \
0       VER          1     SOFT         4      1       99.018997   
1       VER          2     SOFT         5      1       97.973999   
2       VER          3     SOFT         6      1       98.005997   
3       VER          4     SOFT         7      1       97.975998   
4       VER          5     SOFT         8      1       98.035004   
...     ...        ...      ...       ...    ...             ...   
1050    PIA          8     SOFT         8      1      101.294998   
1051    PIA          9     SOFT         9      1      101.533997   
1052    PIA         10     SOFT        10      1      101.584000   
1053    PIA         11     SOFT        11      1      101.351997   
1054    PIA         12     SOFT        12      1      101.155998   

      FuelCorrectedLapTime  NormalizedLapTime  DeltaLapTime  
0                98.978996              7.223      

core           INFO 	Loading data for Bahrain Grand Prix - Race [v3.8.1]
req            INFO 	No cached data found for session_info. Loading data...
_api           INFO 	Fetching session info data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for driver_info. Loading data...
_api           INFO 	Fetching driver list...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for session_status_data. Loading data...
_api           INFO 	Fetching session status data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for lap_count. Loading data...
_api           INFO 	Fetching lap count data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for track_status_data. Loading data...
_api           INFO 	Fetching track status data...
req            INFO 	Data has been written to cache!
req            INFO 	No ca

Error in 2024: Cannot save file into a non-existent directory: 'data'
     Driver  LapNumber Compound  TyreLife  Stint  LapTimeSeconds  \
0       VER          1     SOFT         4      1       97.283997   
1       VER          2     SOFT         5      1       96.295998   
2       VER          3     SOFT         6      1       96.752998   
3       VER          4     SOFT         7      1       96.647003   
4       VER          5     SOFT         8      1       97.172997   
...     ...        ...      ...       ...    ...             ...   
1124    SAR         51     SOFT        11      4       95.972000   
1125    SAR         52     SOFT        12      4       95.987000   
1126    SAR         53     SOFT        13      4       96.087997   
1127    SAR         54     SOFT        14      4       99.613998   
1128    SAR         55     SOFT        15      4       96.209000   

      FuelCorrectedLapTime  NormalizedLapTime  DeltaLapTime  
0                97.244003              6.196      

core           INFO 	Loading data for Bahrain Grand Prix - Race [v3.8.1]
req            INFO 	No cached data found for session_info. Loading data...
_api           INFO 	Fetching session info data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for driver_info. Loading data...
_api           INFO 	Fetching driver list...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for session_status_data. Loading data...
_api           INFO 	Fetching session status data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for lap_count. Loading data...
_api           INFO 	Fetching lap count data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for track_status_data. Loading data...
_api           INFO 	Fetching track status data...
req            INFO 	Data has been written to cache!
req            INFO 	No ca

Error in 2025: Cannot save file into a non-existent directory: 'data'
     Driver  LapNumber Compound  TyreLife  Stint  LapTimeSeconds  \
0       PIA          1     SOFT         4      1       98.693001   
1       PIA          2     SOFT         5      1       97.491997   
2       PIA          3     SOFT         6      1       98.083000   
3       PIA          4     SOFT         7      1       98.133003   
4       PIA          5     SOFT         8      1       98.042999   
...     ...        ...      ...       ...    ...             ...   
1123    HUL         53   MEDIUM        26      3       98.260002   
1124    HUL         54   MEDIUM        27      3       98.498001   
1125    HUL         55   MEDIUM        28      3       98.424004   
1126    HUL         56   MEDIUM        29      3       98.998001   
1127    HUL         57   MEDIUM        30      3       98.822998   

      FuelCorrectedLapTime  NormalizedLapTime  DeltaLapTime  
0                98.653000              4.953      

C:\Users\arnav\AppData\Local\Temp\ipykernel_20676\4068498922.py:29: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  laps['FuelCorrectedLapTime'] = (
C:\Users\arnav\AppData\Local\Temp\ipykernel_20676\4068498922.py:34: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  laps['NormalizedLapTime'] = laps['FuelCorrectedLapTime'] - fastest
C:\Users\arnav\AppData\Local\Temp\ipykernel_20676\4068498922.py:37: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_index

location of the stored file


In [4]:
import os

file_path = f"data/{track.lower()}_{year}.parquet"
print(os.path.abspath(file_path))

c:\DevProjects\Race Strategy Optimization\Code\backend\features\data\bahrain_2025.parquet
